# ISS decoding with PoSTcode

This notebook runs the PoSTcode decoder through the `ISS_decoding` pipeline. Registration, filtering, normalization, and spot detection are still performed by Starfish; only barcode decoding is handed to PoSTcode.

Start with one representative region. After inspecting the results, change `REGIONS_TO_PROCESS` to include additional regions or set it to `None` for all regions.

## Expected inputs and the two SpaceTx paths

`REGIONS_ROOT` must contain region folders named `R1`, `R2`, and so on. The notebook supports either of these starting points:

1. **Build SpaceTx here:** start from `R#/preprocessing/CycleX/4_retiled/` (or its `CARE/` subfolder) plus the original codebook CSV. Set `BUILD_SPACETX = True`.
2. **Use existing SpaceTx:** set `BUILD_SPACETX = False`. Each selected region must already contain `decoding/1_SpaceTX_format/experiment.json` and `codebook.json`. If that tree is stored away from `REGIONS_ROOT`, set `SPACETX_OUTPUT_ROOT` to its parent directory. If there is no separate preprocessing tree, set `REGIONS_ROOT` directly to the existing SpaceTx region tree and leave `SPACETX_OUTPUT_ROOT = None`.

When `SPACETX_OUTPUT_ROOT` is set, the current pipeline reads SpaceTx data and writes decoding outputs below that same alternate root.

## Imports and environment check

In [ ]:
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from starfish import Experiment

import ISS_decoding.SpaceTx_format as STX
import ISS_decoding.decoding as DEC
import ISS_decoding.qc_metrics as QC
from ISS_decoding.postcode_adapter import format_spacetx_codebook_for_postcode

print(f"ISS_decoding: {version('ISS-decoding')}")
print(f"PoSTcode: {version('postcode')}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
print(f"Pinned PoSTcode commit: {DEC.POSTCODE_COMMIT}")

## Configuration

Edit this cell before running the remaining cells. `REGIONS_TO_PROCESS = [1]` is the recommended first test.

The channel names and `DECODING_CHANNELS` order must match the acquisition and the numeric channel codes in the codebook CSV. `PIXEL_TO_UM = 1.0` keeps coordinates in pixels; use the microscope pixel size to report physical coordinates in microns.

In [ ]:
# Parent directory containing R1, R2, ...
REGIONS_ROOT = Path("/path/to/experiment")
REGIONS_TO_PROCESS = [1]

# Path A: build SpaceTx from retiled TIFFs and this headerless codebook CSV.
BUILD_SPACETX = False
CODEBOOK_CSV = Path("/path/to/codebook.csv")
USE_CARE_TILES = True
PIXEL_TO_UM = 1.0
CHANNELS = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"]
DECODING_CHANNELS = ["AF750", "AF488", "Cy3", "Cy5", "At425"]
NUCLEI_CHANNEL = "DAPI"

# Path B: use existing SpaceTx. Leave as None when SpaceTx is below REGIONS_ROOT.
# Otherwise, point this to a tree containing R#/decoding/1_SpaceTX_format/.
SPACETX_OUTPUT_ROOT = None  # or Path("/path/to/existing_spacetx_tree")

# The expensive cells below are guarded so configuration can be checked first.
RUN_DECODING = False

## Optional: create SpaceTx from retiled TIFFs

This is the same formatting route used by the Starfish decoding notebook. It is skipped when `BUILD_SPACETX` is false. Existing `experiment.json` and `codebook.json` files are not overwritten by `make_spacetx_format`.

In [ ]:
if BUILD_SPACETX:
    if not REGIONS_ROOT.exists():
        raise FileNotFoundError(f"REGIONS_ROOT does not exist: {REGIONS_ROOT}")
    if not CODEBOOK_CSV.is_file():
        raise FileNotFoundError(f"CODEBOOK_CSV does not exist: {CODEBOOK_CSV}")

    STX.make_spacetx_format(
        input_dir=REGIONS_ROOT,
        codebook_csv=CODEBOOK_CSV,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=SPACETX_OUTPUT_ROOT,
        pixel_to_um=PIXEL_TO_UM,
        channels=CHANNELS,
        DO_decorators=DECODING_CHANNELS,
        nuclei_channel=NUCLEI_CHANNEL,
        CARE=USE_CARE_TILES,
    )
else:
    print("Skipping SpaceTx generation and using existing formatted data.")

## Validate the SpaceTx experiments and codebooks

This check runs before decoding. It confirms that each selected experiment is present and that its SpaceTx codebook can be converted to PoSTcode's `barcodes × channels × rounds` one-hot array.

In [ ]:
if not REGIONS_ROOT.exists():
    raise FileNotFoundError(f"REGIONS_ROOT does not exist: {REGIONS_ROOT}")

available_regions = sorted(
    int(path.name[1:])
    for path in REGIONS_ROOT.glob("R*")
    if path.is_dir() and path.name[1:].isdigit()
)
selected_region_numbers = available_regions if REGIONS_TO_PROCESS is None else REGIONS_TO_PROCESS
if not selected_region_numbers:
    raise RuntimeError("No regions were selected or discovered.")

data_root = Path(SPACETX_OUTPUT_ROOT) if SPACETX_OUTPUT_ROOT is not None else REGIONS_ROOT
validated_experiments = {}
for region_number in selected_region_numbers:
    region = f"R{region_number}"
    experiment_json = data_root / region / "decoding" / "1_SpaceTX_format" / "experiment.json"
    if not experiment_json.is_file():
        raise FileNotFoundError(f"Missing SpaceTx experiment: {experiment_json}")

    experiment = Experiment.from_json(str(experiment_json))
    barcodes, target_names = format_spacetx_codebook_for_postcode(experiment.codebook)
    validated_experiments[region] = experiment_json
    print(
        f"{region}: {len(list(experiment.keys()))} tiles, "
        f"{barcodes.shape[0]} targets, {barcodes.shape[1]} channels, "
        f"{barcodes.shape[2]} rounds"
    )

validated_experiments

## Configure PoSTcode decoding

`PROBABILITY_THRESHOLD` only controls acceptance into `target`; every detected spot and its probabilities remain in the output. `device='auto'` selects CUDA when available and otherwise uses CPU. Set `SAVE_POSTCODE_ARTIFACTS = True` only when full posterior and fitted-model files are needed, because they can be large.

In [ ]:
PROBABILITY_THRESHOLD = 0.70
SAVE_POSTCODE_ARTIFACTS = False

POSTCODE_KWARGS = {
    "num_iter": 60,
    "batch_size": 15000,
    "up_prc_to_remove": 99.95,
    "modify_bkg_prior": True,
    "estimate_bkg": True,
    "add_remaining_barcodes_prior": 0.05,
    "print_training_progress": True,
    "set_seed": 1,
    "device": "auto",
}

DECODING_KWARGS = {
    "register": False,
    "register_dapi": False,
    "masking_radius": 7,
    "normalization_method": "MH",
    "spot_detection_mode": "starfish",
    "int_threshold": 0.002,
    "sigma_vals": (1, 10, 30),
}

## Run one-region decoding

Set `RUN_DECODING = True` in the configuration cell after the validation output looks correct. Completed tile Parquet files are reused on restart. A completed region is skipped when its canonical region Parquet already exists.

In [ ]:
if RUN_DECODING:
    DEC.process_experiment(
        input_dir=REGIONS_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=SPACETX_OUTPUT_ROOT,
        decode_mode="POSTCODE",
        dense=False,
        prob_threshold=PROBABILITY_THRESHOLD,
        postcode_kwargs=POSTCODE_KWARGS,
        save_postcode_artifacts=SAVE_POSTCODE_ARTIFACTS,
        **DECODING_KWARGS,
    )
else:
    print("Decoding is disabled. Set RUN_DECODING = True when ready.")

## Load and inspect a decoded region

Parquet is the canonical output and preserves the per-round QC arrays. `target` contains accepted genes only; `candidate_target` is the most likely gene even when background or infeasible wins. `assignment_class` records that raw winning class.

In [ ]:
REGION_TO_INSPECT = f"R{selected_region_numbers[0]}"
decoded_dir = data_root / REGION_TO_INSPECT / "decoding" / "2_decoded_postcode"
decoded_file = decoded_dir / f"{REGION_TO_INSPECT}_decoded_postcode.parquet"
if not decoded_file.is_file():
    raise FileNotFoundError(
        f"Decoded output not found: {decoded_file}. Run the decoding cell first."
    )

reads = pd.read_parquet(decoded_file)
print(f"Loaded {len(reads):,} detected spots from {decoded_file}")
display(reads.head())

In [ ]:
assignment_summary = (
    reads.groupby(["assignment_class", "passes_thresholds"], dropna=False)
    .size()
    .rename("spots")
    .reset_index()
)
display(assignment_summary)

accepted_reads = reads.loc[reads["passes_thresholds"]].copy()
print(f"Accepted gene assignments: {len(accepted_reads):,} / {len(reads):,}")
display(accepted_reads["target"].value_counts().head(20).rename("spots"))

## Re-threshold without decoding again

Because all detected spots are retained, a stricter gene-probability threshold can be explored without re-running PoSTcode. A spot is accepted only when the raw winning class is `gene` and its winning probability passes the new threshold.

In [ ]:
ANALYSIS_THRESHOLD = 0.80
analysis_mask = (
    reads["assignment_class"].eq("gene")
    & reads["assignment_probability"].ge(ANALYSIS_THRESHOLD)
)
analysis_reads = reads.loc[analysis_mask].copy()
analysis_reads["target"] = analysis_reads["candidate_target"]
print(f"Accepted at {ANALYSIS_THRESHOLD:.0%}: {len(analysis_reads):,} spots")

## Basic QC and spatial inspection

Parquet stores `quality_all_bases` as arrays. The first cell below expands them into the cycle columns expected by the existing QC plotting helpers.

In [ ]:
qc_reads = reads.copy()
if len(qc_reads):
    quality_per_cycle = np.vstack(qc_reads["quality_all_bases"].to_numpy())
    for cycle_index in range(quality_per_cycle.shape[1]):
        qc_reads[f"qc_cycle{cycle_index + 1}"] = quality_per_cycle[:, cycle_index]

    QC.quality_per_cycle(
        qc_reads,
        cycles=quality_per_cycle.shape[1],
        format_base_quality=True,
    )
    QC.compare_scores(
        qc_reads,
        score1="quality_mean",
        score2="quality_minimum",
        hue="assigned",
        kind="hist",
        format_base_quality=True,
    )
else:
    print("This region contains no detected spots.")

In [ ]:
if len(analysis_reads):
    QC.plot_frequencies(analysis_reads, on="target")
    QC.plot_expression(
        analysis_reads,
        key="target",
        xcolumn="xc",
        ycolumn="yc",
        genes="all",
        size=4,
        background="black",
        title_color="white",
        figuresize=(10, 10),
        save=None,
        fmt="pdf",
    )

## Optional: inspect saved posterior and model artifacts

When `SAVE_POSTCODE_ARTIFACTS` was enabled, `posteriors/fov_*.npz` contains the full class-probability matrix and class mappings. `models/fov_*.npz` contains the fitted parameters, normalization constants, codebook snapshot, settings, and loss history.

In [ ]:
posterior_files = sorted((decoded_dir / "posteriors").glob("fov_*.npz"))
model_files = sorted((decoded_dir / "models").glob("fov_*.npz"))
print(f"Posterior files: {len(posterior_files)}; model files: {len(model_files)}")

if posterior_files:
    with np.load(posterior_files[0], allow_pickle=False) as posterior:
        print(f"Example: {posterior_files[0].name}")
        print(f"class_probs shape: {posterior['class_probs'].shape}")
        print(f"targets: {len(posterior['target_names'])}")
        print(f"first spot UID: {posterior['spot_uid'][0] if len(posterior['spot_uid']) else 'none'}")

## Scale up

After the representative region has been checked, update `REGIONS_TO_PROCESS` in the configuration cell (for example `[1, 2, 3]`) or set it to `None` to process every discovered region. Keep the same PoSTcode seed and parameter dictionary for a comparable run.